[![pythonista.io](img/pythonista.png)](https://www.pythonista.io)

# Proyecto final *end to end*.

Este cierre integra la cadena completa de entrega implementada en el curso: cinco *workflows* de *GitHub Actions*, *quality gates* con `uv`, `ruff`, `mypy` y `pytest`, publicación de imagen en *Docker Hub* y *GHCR* con *SBOM* y firma *Cosign*, despliegue en *Cloud Run v2* vía *OIDC*, e infraestructura gestionada con *Terraform* en *GCP*.

## Objetivos.

- Identificar qué control defiende qué riesgo en la cadena implementada.
- Leer el código real —*workflows*, `Dockerfile`, `pyproject.toml`, `main.tf`— como evidencia de los controles activos.
- Verificar que el *pipeline* completo funciona de extremo a extremo sobre un *tag* semver real.

## La cadena de entrega.

| Archivo | *Trigger* | Qué hace |
|---|---|---|
| `python-app-test.yml` | *Push* a `main`, *pull request*, manual | *Quality gates* en matriz *Python* 3.11/3.12/3.13: `ruff check`, `ruff format --check`, `mypy`, `pytest --cov-fail-under=80`, *Trivy* `fs` → SARIF a *GitHub* Security |
| `empaqueta.yaml` | Manual | *Job* `calidad` (*quality gates*) + *job* `build_artifact` (*tox* en py311/py312/py313 → artefacto `.tar.gz`) |
| `envia-a-docker.yaml` | Manual | *Lint* *Dockerfile* (*Hadolint*), *Trivy* config scan, *build* y *push* a *Docker Hub*, *SBOM* (*Syft*), firma *Cosign* keyless, verificación de firma y attestation |
| `envia-a-packages.yml` | *Tag* `v*.*.*-*` (prerelease), manual | Igual que anterior pero publica en *GHCR*; solo para entornos de dev/test |
| `despliega-cloud-run.yaml` | *Tag* `v*.*.*` (estable), manual | Pipeline completo de prod: *lint* Dockerfile, *Trivy* config, *build* y *push* a *GHCR* + *Artifact Registry*, *Trivy* image scan, *SBOM* + firma *Cosign*, despliegue *blue/green* en *Cloud Run* con smoke test y rollback automático |

## *Quality gates*.

```text
uv sync --frozen --extra dev   ← lockfile con SHA-256
        ↓
ruff check .                   ← lint
        ↓
ruff format --check .          ← formato
        ↓
mypy .                         ← tipado estático
        ↓
pytest --cov-fail-under=80     ← pruebas + cobertura ≥ 80 %
        ↓
trivy fs → SARIF               ← vulnerabilidades en dependencias
```

## Política de *tags* y entornos.

| Patrón | Workflow | Entorno | Ejemplo |
|---|---|---|---|
| `vX.Y.Z` (sin guion) | `despliega-cloud-run.yaml` | `prod` (requiere aprobación manual) | `v1.2.0` |
| `vX.Y.Z-rc.N` | `envia-a-packages.yml` | `test` (automático) | `v1.2.0-rc.1` |
| `vX.Y.Z-beta.N` | `envia-a-packages.yml` | `test` (automático) | `v1.2.0-beta.2` |
| `vX.Y.Z-alpha.N` | `envia-a-packages.yml` | `test` (automático) | `v1.2.0-alpha.1` |

Los tags estables (`vX.Y.Z`) disparan el pipeline completo de prod en `despliega-cloud-run.yaml`: build → firma → deploy. Los tags prerelease solo publican la imagen en *GHCR* sin desplegar. `envia-a-docker.yaml` no tiene trigger automático en tags; se ejecuta solo de forma manual.

## Infraestructura como código.

La infraestructura se gestiona con *Terraform* en `infra/terraform-py271/`. Los recursos declarados en `main.tf` e `iam.tf` son los siguientes:

| Recurso | Descripción |
|---|---|
| `google_artifact_registry_repository` | Repositorio *Docker* en *GCP* para almacenar imágenes |
| `google_sql_database_instance` | Instancia *Cloud SQL* PostgreSQL 15 |
| `google_sql_database` y `google_sql_user` | Base de datos y usuario de la aplicación |
| `google_secret_manager_secret` (`DATABASE_URL`) | Cadena de conexión; valor poblado manualmente tras `apply`, nunca en estado de *Terraform* |
| `google_cloud_run_v2_service` | Servicio con imagen *placeholder*; `lifecycle.ignore_changes` en `image`, `env` y `volume_mounts`; `DATABASE_URL` referenciada vía `value_source.secret_key_ref`; socket *Cloud SQL* montado en `/cloudsql` |
| `google_service_account.cicd_deployer` | SA de *GitHub Actions* (roles: `run.admin`, `artifactregistry.writer`, `iam.serviceAccountUser`, `secretmanager.admin`, etc.) |
| `google_service_account.cloud_run_runtime` | SA de *runtime* (roles: `cloudsql.client`, `secretmanager.secretAccessor` solo sobre `DATABASE_URL`) |
| `google_iam_workload_identity_pool` + `google_iam_workload_identity_pool_provider` | *OIDC* sin llaves estáticas; `attribute_condition` restringe acceso a los repositorios exactos de app e infra |
| `google_service_account_iam_member.cicd_self_token_creator` | Permite al SA de CI/CD generar *identity tokens* vía auto-impersonación para el smoke test |
| `google_cloud_run_v2_service_iam_member.cicd_invoker` | Permite al SA de CI/CD invocar el servicio *Cloud Run* (URL candidata del smoke test) |
| `google_cloud_run_v2_service_iam_member.public_invoker` | Acceso público al servicio (`allUsers`); eliminar para servicio privado |

El *workflow* de *Terraform* en `infra/terraform-py271/.github/workflows/terraform-plan-apply.yml` acepta `workflow_dispatch` con parámetros `target_environment` (test/prod) y `run_apply` (false/true). Los pasos son: `terraform init` con *backend* *GCS*, `fmt -check`, `validate`, `plan` y `apply` condicional.

## Seguridad de cadena de suministro.

- *Hadolint* v2.12.0: *lint* del *Dockerfile* antes del *build*; bloquea malas prácticas como capas innecesarias o imagen base sin versión fija.
- *Trivy* config scan (HIGH/CRITICAL, `exit-code: 1`): escanea el `Dockerfile` antes del *push* y bloquea si hay hallazgos.
- *Trivy* image scan (HIGH/CRITICAL, `ignore-unfixed`, `exit-code: 1`): escanea la imagen por *digest* inmutable tras el *push*.
- *Syft* + *Cosign* keyless (*OIDC*): genera *SBOM* SPDX-JSON, firma la imagen y adjunta la attestation al registro.
- *Dependabot* configurado para tres ecosistemas: `pip` (grupos dev), `github-actions` (grupos por proveedor) y `docker` (*digest* de imagen base).
- `CODEOWNERS`: aprobación del propietario requerida en *PRs* que toquen `.github/workflows/`, `pyproject.toml`, `requirements.txt`, `uv.lock` e `infra/`.

## Despliegue defensivo: *smoke test* y *rollback*.

`despliega-cloud-run.yaml` ejecuta el pipeline completo de prod en un solo *job* condicional:

1. **Build** (solo en tags, no en re-deploy manual): *lint* Dockerfile, *Trivy* config scan, `docker build` + `push` a *GHCR* y *Artifact Registry*, *Trivy* image scan, *SBOM* con *Syft*, firma *Cosign* y attestation.
2. **Captura de revisión activa** antes del deploy (p. ej. `apiflask-demo-00003-abc`). *Cloud Run* no acepta `PREVIOUS` como identificador; hay que capturar el nombre exacto.
3. **Deploy sin tráfico** con `--no-traffic --tag <rollout-tag>`. *Cloud Run* asigna una URL exclusiva a la revisión candidata con el patrón `https://<tag>---<servicio>-<hash>.<region>.run.app`.
4. **Smoke test** con *identity token* por impersonación (las credenciales WIF no generan *identity tokens* directamente):
   ```bash
   curl -H "Authorization: Bearer $(gcloud auth print-identity-token \
         --impersonate-service-account=${{ vars.GCP_SERVICE_ACCOUNT }})" \
     "$CANDIDATE_URL/api/alumnos/"
   ```
   Espera HTTP 200. Un `200 []` (tabla vacía) es suficiente para validar que la app arrancó y conectó a *Cloud SQL*.
5. **Si pasa**: `gcloud run services update-traffic --to-latest` (100 % del tráfico a la nueva revisión).
6. **Si falla**: `gcloud run services update-traffic --to-revisions=<PREV_REVISION>=100` (restaura la revisión anterior exacta).

## Modelo de amenaza.

| Control en el *pipeline* | Riesgo que mitiga |
|---|---|
| *Lint* y tipado estático | Código con errores de tipos o sintaxis llega a producción |
| Pruebas automatizadas con cobertura ≥ 80 % | Regresiones silenciosas introducidas por un *commit* |
| *Build* reproducible con `uv sync --frozen` | *Supply chain drift*: imagen construida con dependencias distintas a las validadas |
| *Lint* de *Dockerfile* (*Hadolint*) | Malas prácticas en la imagen base (capas innecesarias, `latest`, `apt` sin versión) |
| Escaneo de imagen por *digest* (*Trivy*) | CVEs conocidos en capas base o dependencias llegan al servicio |
| *SBOM* + attestation (*Syft* + *Cosign*) | Artefacto sin trazabilidad; sin evidencia forense post-incidente |
| *Dependabot* (pip, actions, *Docker*) | Dependencias con CVE conocido permanecen sin actualizar |
| `CODEOWNERS` sobre *workflows* e `infra/` | Un colaborador modifica el *pipeline* o la *IaC* sin revisión del propietario |
| *Plan* de *Terraform* con aprobación | Cambios de infraestructura no revisados destruyen recursos o abren puertos |
| *OIDC* / *WIF* sin llaves estáticas | Credencial robada permite *deploys* desde fuera del *pipeline* autorizado |
| *Smoke test* + *rollback* automático | *Deploy* roto queda activo porque nadie lo verificó post-*deploy* |
| `DATABASE_URL` en *Secret Manager* | Contraseña de base de datos expuesta en variables de entorno o estado de *Terraform* |

## Cierre pedagógico.

El recorrido de este curso parte de un riesgo concreto —un *pipeline* sin controles— y añade una capa de defensa en cada sesión. El proyecto final no es un ejercicio de integración técnica: es la demostración de que cada control tiene un adversario específico y una posición justificada en el flujo de entrega.

Cada archivo del repositorio —`python-app-test.yml`, `envia-a-packages.yml`, `despliega-cloud-run.yaml`, `main.tf`, `iam.tf`, `CODEOWNERS`— es un control activo. Quitarlo tiene un coste en riesgo que se puede nombrar.

Un *pipeline* defendible no es el más rápido ni el más simple; es el que puede explicar, para cada etapa, qué rompe si la quitas.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>